# House Price Forecast

## 1. Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import shapiro
from statsmodels.tools import eval_measures

Source of data: https://www.kaggle.com/harlfoxem/housesalesprediction  
link -> Discussion -> Column definitions

--- 
### TASKS
TASK1: Load data, familiarize yourself with it (missing values, outliers), display dependency plots and calculate correlations.

TASK2: Build a linear regression model, try to achieve a high R^2 score, examine residuals.

---

## STEP 1: Import Data

In [ ]:
try:
    houses = pd.read_csv("data/kc_house_data.csv")
except FileNotFoundError:
    try:
        houses = pd.read_csv("../data/kc_house_data.csv")
    except FileNotFoundError:
        print("File not found. Make sure you are in the correct directory.")
        houses = pd.DataFrame() # Empty dataframe to avoid errors

### Data Overview

In [ ]:
if not houses.empty:
    print("Info:")
    print(houses.info())
    print("\nDescribe:")
    print(houses.describe())
    print("\nMissing Data:")
    print(houses.isnull().sum())

### Correlation and Plot (Price vs Area)

In [ ]:
if not houses.empty:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='sqft_living', y='price', data=houses)
    plt.title('Price vs Living Area (sqft_living)')
    plt.show()

    corr = houses['price'].corr(houses['sqft_living'])
    print(f"\nCorrelation price vs sqft_living: {corr}")

## STEP 2: Data Wrangling

In [ ]:
if not houses.empty:
    houses['date'] = pd.to_datetime(houses['date'])
    houses['yr_sold'] = houses['date'].dt.year
    houses['age'] = houses['yr_sold'] - houses['yr_built']
    # Correction of possible negative age values (data errors or renovations treated as construction)
    houses['age'] = houses['age'].apply(lambda x: x if x >= 0 else 0)

    # Additional variable - luxury districts
    # Define luxury districts as those in the upper quartile of average prices
    zip_price = houses.groupby('zipcode')['price'].mean()
    luxury_threshold = zip_price.quantile(0.75)
    luxury_zips = zip_price[zip_price > luxury_threshold].index
    
    houses['is_luxury'] = houses['zipcode'].apply(lambda x: 1 if x in luxury_zips else 0)

### Data Visualization (Price vs Grade)

In [ ]:
if not houses.empty:
    plt.figure(figsize=(10,6))
    sns.boxplot(x='grade', y='price', data=houses)
    plt.title("Price vs Construction Grade")
    plt.show()

## STEP 4: Model Training

In [ ]:
if not houses.empty:
    # Model 1
    # We choose variables that seem significant: sqft_living, grade, age, waterfront
    formula1 = "price ~ sqft_living + grade + age + C(waterfront)"
    model1 = smf.ols(formula=formula1, data=houses)
    result1 = model1.fit()

    # Model 2
    # We try to improve the result by logging the price (common practice with real estate prices)
    # and adding luxury and view variables
    houses['log_price'] = np.log(houses['price'])
    
    formula2 = "log_price ~ sqft_living + grade + age + C(waterfront) + is_luxury + view + bathrooms"
    model2 = smf.ols(formula=formula2, data=houses)
    result2 = model2.fit()

    # Model 3 - Improved
    # Adding interactions and location variables and surroundings (sqft_living15)
    # grade squared or as an important non-linear predictor
    formula3 = "log_price ~ sqft_living + I(sqft_living**2) + grade + age + C(waterfront) + is_luxury + view + bathrooms + lat + sqft_living15 + sqft_living:grade"
    model3 = smf.ols(formula=formula3, data=houses)
    result3 = model3.fit()

## STEP 5: Model Evaluation

In [ ]:
if not houses.empty:
    # print(result1.summary())
    # print(result2.summary())

    ## model 1
    forecast_model1 = result1.fittedvalues
    
    plt.figure(figsize=(8,6))
    plt.scatter(houses['price'], forecast_model1, alpha=0.5)
    plt.plot([houses['price'].min(), houses['price'].max()], [houses['price'].min(), houses['price'].max()], 'r--')
    plt.title("Model 1: Empirical vs Forecast")
    plt.xlabel("Empirical Price")
    plt.ylabel("Forecasted Price")
    plt.show()

    ## model 2
    # Model 2 forecasts log, so we need to do exp() to compare
    forecast_model2_log = result2.fittedvalues
    forecast_model2 = np.exp(forecast_model2_log)
    
    # Residuals Histogram (Log Scale)
    plt.figure(figsize=(8,6))
    result2.resid.hist(bins=50) 
    plt.title("Model 2 Residuals Histogram (Log Scale)")
    plt.show()
    
    ## model 3
    forecast_model3_log = result3.fittedvalues
    forecast_model3 = np.exp(forecast_model3_log)

### Tests and Grades

In [ ]:
if not houses.empty:
    # We sample, because for large data Shapiro always rejects H0
    sample_resid = result2.resid.sample(2000) if len(result2.resid) > 2000 else result2.resid
    stat, p = shapiro(sample_resid)
    print(f"\nShapiro-Wilk Test (Model 2 residuals): p-value = {p}")

    def mape(y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    rmse1 = eval_measures.rmse(houses['price'], forecast_model1)
    rmse2 = eval_measures.rmse(houses['price'], forecast_model2)
    rmse3 = eval_measures.rmse(houses['price'], forecast_model3)
    
    mape1 = mape(houses['price'], forecast_model1)
    mape2 = mape(houses['price'], forecast_model2)
    mape3 = mape(houses['price'], forecast_model3)

    print("\n" + "="*70)
    print(f"{'Metric':<15} | {'Model 1':<15} | {'Model 2':<15} | {'Model 3 (New)':<15}")
    print("-" * 70)
    print(f"{'R-squared':<15} | {result1.rsquared:<15.4f} | {result2.rsquared:<15.4f} (log) | {result3.rsquared:<15.4f} (log)")
    print(f"{'RMSE':<15} | {rmse1:<15.2f} | {rmse2:<15.2f} | {rmse3:<15.2f}")
    print(f"{'MAPE (%)':<15} | {mape1:<15.2f} | {mape2:<15.2f} | {mape3:<15.2f}")
    print(f"{'AIC':<15} | {result1.aic:<15.2f} | {result2.aic:<15.2f} | {result3.aic:<15.2f}")
    print("="*70 + "\n")

## STEP 6: Prediction for new data

In [ ]:
if not houses.empty:
    print("\n" + "="*90)
    print("PRICE PREDICTION FOR VARIOUS HOUSES (MODEL 3)")
    print("-" * 90)

    # 1. Define data for several houses with different profiles
    # Mean values from dataset to fill missing (less important) variables
    mean_lat = houses['lat'].mean()
    mean_sqft15 = houses['sqft_living15'].mean()
    
    new_houses_data = {
        'description': [
            'Small house, standard', 
            'Large family house, high standard', 
            'Waterfront villa, luxury', 
            'Old shack for renovation', 
            'City apartment (hypothetical)'
        ],
        'sqft_living': [1000, 2500, 4500, 1200, 1800],
        'grade': [7, 9, 11, 5, 8],          # 1-13 scale (7 is avg, 11 is excellent)
        'age': [10, 5, 2, 60, 15],
        'waterfront': [0, 0, 1, 0, 0],      # 0/1
        'is_luxury': [0, 0, 1, 0, 1],       # 0/1 (based on zipcode)
        'view': [0, 2, 4, 0, 1],            # 0-4 scale
        'bathrooms': [1, 2.5, 4.5, 1, 2],
        'lat': [mean_lat, mean_lat, mean_lat + 0.1, mean_lat - 0.05, mean_lat + 0.05], # Location matters
        'sqft_living15': [1200, 2400, 4000, 1200, 2000] # Surroundings
    }

    new_houses_df = pd.DataFrame(new_houses_data)

    # 2. Prediction (Model 3 returns log -> we do exp)
    predicted_log = result3.predict(new_houses_df)
    new_houses_df['predicted_price'] = np.exp(predicted_log)

    # Display results
    print(f"{'House Description':<35} | {'Area (sqft)':<12} | {'Grade':<6} | {'Water':<5} | {'Predicted Price ($)':<25}")
    print("-" * 90)

    for index, row in new_houses_df.iterrows():
        print(f"{row['description']:<35} | {row['sqft_living']:<12} | {row['grade']:<6} | {row['waterfront']:<5} | {row['predicted_price']:<15,.2f}")

    print("="*90 + "\n")